In [1]:
# Importar librerías
import pandas as pd
from src.config import data_folder
from src.funcionesExtract import *

In [2]:
# Obtener lista de tickers del S&P 500 desde el fichero constituents.csv
# Si no existe el fichero, lo descarga desde GitHub
# Para actualizar la lista de componentes, cambiar force_update a True, ejecutar y volver a dejar en False
ruta_archivo = descargar_constituents(force_update=False) 

# Cargar datos
df_tickers = pd.read_csv(ruta_archivo)
df_tickers.info()

Descargando constituents.csv desde GitHub...
Descarga completada.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Symbol                 503 non-null    object
 1   Security               503 non-null    object
 2   GICS Sector            503 non-null    object
 3   GICS Sub-Industry      503 non-null    object
 4   Headquarters Location  503 non-null    object
 5   Date added             503 non-null    object
 6   CIK                    503 non-null    int64 
 7   Founded                503 non-null    object
dtypes: int64(1), object(7)
memory usage: 31.6+ KB


In [3]:
# Limpieza previa de df_tickers
df_tickers_clean = limpieza_tickers(df_tickers)

# Lista de tickers
tickers_list = df_tickers_clean["Ticker"].tolist()
df_tickers_clean.head()

,Ticker,Sector,DateAdded
0,MMM,Industrials,1957-03-04
1,AOS,Industrials,2017-07-26
2,ABT,HealthCare,1957-03-04
3,ABBV,HealthCare,2012-12-31
4,ACN,InformationTechnology,2011-07-06


In [4]:
# Extraer precios de los tickers y del índice SPY (demora unos 3 minutos)
df_prices = extraer_precios(tickers_list)

# Se extrae precio del Índice de Mercado para usar en cálculos y se guarda en fichero
df_index = extraer_precios(['SPY'])
df_index.to_parquet(f"{data_folder}/market_index.parquet")

df_prices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23950 entries, 0 to 23949
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       23950 non-null  datetime64[ns]
 1   Close      23950 non-null  float64       
 2   Dividends  23950 non-null  float64       
 3   Ticker     23950 non-null  object        
dtypes: datetime64[ns](1), float64(2), object(1)
memory usage: 748.6+ KB


In [5]:
df_index.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       48 non-null     datetime64[ns]
 1   Close      48 non-null     float64       
 2   Dividends  48 non-null     float64       
 3   Ticker     48 non-null     object        
dtypes: datetime64[ns](1), float64(2), object(1)
memory usage: 1.6+ KB


In [6]:
# Unir df_prices y df_tickers
df_merged = pd.merge(
    df_prices,
    df_tickers_clean,
    on= 'Ticker',
    how= 'left'
)
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23950 entries, 0 to 23949
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       23950 non-null  datetime64[ns]
 1   Close      23950 non-null  float64       
 2   Dividends  23950 non-null  float64       
 3   Ticker     23950 non-null  object        
 4   Sector     23950 non-null  object        
 5   DateAdded  23950 non-null  object        
dtypes: datetime64[ns](1), float64(2), object(3)
memory usage: 1.1+ MB


In [7]:
# Extraer datos financieros de yfinance (demora unos 10 minutos)
df_financials = extraer_financials(tickers_list)
df_financials.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2380 entries, 0 to 2379
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Date                       2380 non-null   datetime64[ns]
 1   Total Revenue              2010 non-null   float64       
 2   Gross Profit               1800 non-null   float64       
 3   Operating Income           1826 non-null   float64       
 4   Net Income                 2002 non-null   float64       
 5   EBITDA                     1822 non-null   float64       
 6   Basic Average Shares       2003 non-null   float64       
 7   Cash And Cash Equivalents  2002 non-null   float64       
 8   Current Debt               1602 non-null   float64       
 9   Long Term Debt             1892 non-null   float64       
 10  Total Debt                 1990 non-null   float64       
 11  Stockholders Equity        2002 non-null   float64       
 12  Total 

In [8]:
# Unir dataframe con datos financieros
df_final = pd.merge(
    df_merged, 
    df_financials, 
    on=['Date', 'Ticker'],
    how='left'
)
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23950 entries, 0 to 23949
Data columns (total 25 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Date                       23950 non-null  datetime64[ns]
 1   Close                      23950 non-null  float64       
 2   Dividends                  23950 non-null  float64       
 3   Ticker                     23950 non-null  object        
 4   Sector                     23950 non-null  object        
 5   DateAdded                  23950 non-null  object        
 6   Total Revenue              1992 non-null   float64       
 7   Gross Profit               1782 non-null   float64       
 8   Operating Income           1808 non-null   float64       
 9   Net Income                 1984 non-null   float64       
 10  EBITDA                     1804 non-null   float64       
 11  Basic Average Shares       1983 non-null   float64       
 12  Cash

In [9]:
# Limpieza final, con forward fill para datos financieros
df_final_clean = limpieza_final(df_final)
df_final_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18016 entries, 0 to 18015
Data columns (total 25 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Date                    18016 non-null  datetime64[ns]
 1   Close                   18016 non-null  float64       
 2   Dividends               18016 non-null  float64       
 3   Ticker                  18016 non-null  object        
 4   Sector                  18016 non-null  object        
 5   DateAdded               18016 non-null  object        
 6   TotalRevenue            18016 non-null  float64       
 7   GrossProfit             17755 non-null  float64       
 8   OperatingIncome         18016 non-null  float64       
 9   NetIncome               17977 non-null  float64       
 10  EBITDA                  18016 non-null  float64       
 11  BasicAverageShares      17965 non-null  float64       
 12  CashAndCashEquivalents  17977 non-null  float6

In [10]:
# Guardar datos extraidos en fichero raw_data
df_final_clean.to_parquet(f"{data_folder}/raw_data.parquet")
print("Fichero guardado en la carpeta",data_folder)

Fichero guardado en la carpeta data
